## <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Curs 4b: Regularizare: Ridge, Lasso, Elastic Net</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/04-regresie/curs-04b-regularizarea-regresiei.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "04-regresie"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Cum lucrăm azi: tu, agentul și notebook-ul

Acest curs se joacă în trei: **lecția** (acest notebook), **tu** și **agentul tău** (Gemini în Colab, un agent din VS Code sau din Antigravity). Agentul scrie codul; tu decizi ce se cere, de ce, și verifici dacă rezultatul are sens. Fiecare idee este un **capitol** cu aceiași pași:

| Pas | Ce faci |
|---|---|
| **Concept** | citești 2-3 propoziții |
| **Prezice** | spui (sau scrii) ce crezi că va ieși, *înainte* să rulezi |
| **Cere agentului** | formulezi cererea; apoi rulezi codul de referință și îl compari cu ce a scris agentul |
| **Verifică** | compari cu valorile de control |
| **Capcană** | vezi ce poate merge prost, chiar dacă codul rulează |
| **Pentru client** | spui rezultatul în limbaj de manager |

**Primer** (lipește-l agentului la începutul conversației; dacă ai pierdut conversația sau agentul a rămas fără tokeni, lipește primerul și *Jurnalul de sesiune* de la finalul notebook-ului unui alt agent și scrie „continuă de aici”):

```text
Rolul tău: ești partenerul meu de lucru la cursul SEED1 (analiză de date și învățare automată), modulul 4: regresie (regresie liniară simplă și multiplă, supraînvățare, regularizare Ridge, Lasso, Elastic Net). Eu sunt student și nu știu să programez în Python; eu decid ce se face și de ce, tu scrii codul, iar eu verific rezultatul.

Datele cursului:
- Se încarcă DOAR cu `from seed.date import incarca`. `incarca("lab-set6")` întoarce un dicționar nume_tabel -> pandas DataFrame; tabelul cu prețuri se ia cu `incarca("lab-set6", "train")`. Tabelul `test` NU are coloana țintă `SalePrice`.
- Nu folosi căi de fișiere și nu descărca date de pe internet.
- Folosește pandas, matplotlib/seaborn și scikit-learn. Cod scurt, comentarii în română, un pas pe rând.

Reguli pentru acest modul (le aplici chiar dacă nu le cer eu):
1. Scoate ID-urile (`Id`) și ținta din predictori. Împarte train/test (20% test, `random_state=42`) ÎNAINTE de orice transformare învățată din date (completarea lipsurilor, one-hot, scalare).
2. Pune pregătirea și modelul într-un `Pipeline`, ca transformările să fie învățate doar pe antrenare.
3. Compară întotdeauna cu modelul de referință (media) și raportează MAE, RMSE și R² pe antrenare ȘI pe test.
4. Hiperparametrii (de exemplu `alpha`) se aleg prin validare încrucișată pe antrenare, niciodată după scorul pe test.
5. Nu inventa coloane: dacă nu ești sigur, afișează `df.columns`.
6. Spune-mi dacă ceva din rezultat îți pare ciudat (diferență mare între antrenare și test, coeficienți cu semn absurd), în loc să-l prezinți ca pe un succes.

Cum lucrăm:
- Implicit ești în MOD TUTORE: înainte de a-mi da codul, îmi pui o singură întrebare scurtă despre concept („ce crezi că se va întâmpla?”). Nu rezolvi temele în locul meu; mă ajuți să verific.
- La cererea mea poți trece în MOD EXECUTANT (execuți direct și spui pe scurt ce ai făcut) sau în MOD RECENZENT (cauți greșeli în ce îți arăt: ID-uri printre predictori, evaluare pe datele de antrenare, transformări învățate înainte de împărțire).
- Când scriu „închide capitolul”, completezi Jurnalul de sesiune în formatul de mai jos, numai din ce am discutat. Dacă ceva nu s-a discutat, scrii „nu știu”.

Formatul Jurnalului de sesiune:
## Jurnal de sesiune
- Modul / capitolul la care am ajuns:
- Agent folosit și data:
- Ce am decis și de ce:
- Valori de control verificate (da / nu / care):
- Greșeli găsite la agent:
- Întrebare deschisă:
- Pentru următorul agent (ce a rămas de făcut, într-o propoziție):

Răspunde acum doar cu: rolul tău în două propoziții și cele 6 reguli, pe scurt.
```

## Unde am rămas

În `curs-04a` am văzut că un model cu multe coloane și puține case **memorează** ($R^2$ = 1 pe antrenare, slab pe test). Azi îl ținem în frâu cu **regularizarea**, alegem cât de tare prin **validare încrucișată** și formulăm răspunsul pentru agenție: *cât cerem și cât de mult putem greși*.

**Planul de azi (≈50 min):** capitolul 5 o singură împărțire minte · 6 Ridge · 7 Lasso · 8 Elastic Net și clasamentul · ce îi spunem clientului.

Celula de mai jos refă pașii din `curs-04a` (date, împărțire, recuzita `pregatire()`) și adaugă funcția **`evalueaza`**: antrenează orice model și îi adaugă un rând într-un **clasament**. Schimbarea modelului devine o singură linie; tot restul (împărțirea, metricile) rămâne identic, deci comparația e cinstită. Aceeași idee o folosim în modulul 5 pentru clasificatori.

In [ ]:
# Unde am rămas în curs-04a: date, împărțire, pregătire (rulați această celulă)
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from seed.date import incarca
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

df = incarca("lab-set6", "train")
X = df.drop(columns=["Id", "SalePrice"])
y = df["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def pregatire(scalare=False):
    """Rețeta de pregătire, învățată DOAR pe antrenare (în Pipeline).
    Numere: mediana în locul valorilor lipsă. Text: „Lipsă” în locul valorilor lipsă, apoi one-hot
    (categoriile noi din test sunt ignorate). La cerere: standardizarea tuturor coloanelor rezultate."""
    text = make_pipeline(SimpleImputer(strategy="constant", fill_value="Lipsă"),
                         OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    coloane = make_column_transformer((SimpleImputer(strategy="median"), make_column_selector(dtype_include="number")),
                                      (text, make_column_selector(dtype_exclude="number")),
                                      verbose_feature_names_out=False)
    return make_pipeline(coloane, StandardScaler()) if scalare else coloane

clasament = []

def evalueaza(model, nume, X_antrenare=X_train, y_antrenare=y_train):
    """Antrenează modelul și adaugă în clasament erorile pe antrenare și pe test (același test pentru toți)."""
    start = time.time()
    model.fit(X_antrenare, y_antrenare)
    p_antr, p_test = model.predict(X_antrenare), model.predict(X_test)
    clasament.append({"model": nume,
                      "MAE antrenare": round(mean_absolute_error(y_antrenare, p_antr)),
                      "MAE test": round(mean_absolute_error(y_test, p_test)),
                      "R2 antrenare": round(r2_score(y_antrenare, p_antr), 3),
                      "R2 test": round(r2_score(y_test, p_test), 3),
                      "secunde": round(time.time() - start, 1)})
    return model

evalueaza(DummyRegressor(strategy="mean"), "referință: media")
evalueaza(make_pipeline(pregatire(), LinearRegression()), "liniară, toate casele")
pd.DataFrame(clasament)

**Verifică.** Media: MAE pe test ≈ 62.576 $. Regresia liniară cu toate coloanele: $R^2$ 0,937 pe antrenare și 0,887 pe test, MAE ≈ 18.665 $ (ca la finalul capitolului 4).

## Capitolul 5. O singură împărțire minte

**Concept.** Scorul pe test depinde de **ce case** au căzut în test. O singură împărțire poate fi norocoasă sau ghinionistă. **Validarea încrucișată** (*cross-validation*) împarte antrenarea în $k$ părți (de obicei 5) și antrenează de 5 ori, de fiecare dată pe 4 părți, testând pe a cincea; raportăm media și împrăștierea celor 5 scoruri. Azi o folosim „din cutie”, ca să alegem cât de tare regularizăm; în modulul 6 o studiem pe larg.

**Prezice.** Aceeași regresie, aceleași date; schimbăm doar împărțirea aleatoare (`random_state` de la 0 la 4). Cât de mult crezi că variază $R^2$ pe test?

**Cere agentului:**

> Pentru `random_state` de la 0 la 4, refă împărțirea 80/20 și antrenează aceeași regresie liniară pe coloanele numerice (lipsurile completate cu mediana, în `Pipeline`). Afișează R² pe test pentru fiecare. Apoi fă o validare încrucișată cu 5 părți pe antrenarea inițială.

In [ ]:
# Cod de referință: același model, cinci împărțiri diferite
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score

numerice = X.select_dtypes("number").columns
regresie = make_pipeline(SimpleImputer(strategy="median"), LinearRegression())
for rs in range(5):
    a, b, c, d = train_test_split(X[numerice], y, test_size=0.2, random_state=rs)
    print(f"random_state={rs}:  R2 test = {regresie.fit(a, c).score(b, d):.3f}")

scoruri = cross_val_score(regresie, X_train[numerice], y_train, cv=5, scoring="r2")
print("\nValidare încrucișată (5 părți din antrenare):", scoruri.round(3), "  media:", round(scoruri.mean(), 3))

**Verifică.** $R^2$ pe test variază între ≈ 0,50 (`random_state=3`) și 0,85 (`random_state=2`) doar din cauza împărțirii; în validarea încrucișată, cele 5 scoruri merg de la 0,59 la 0,87 (media 0,76).

**Capcană.** Dacă cele câteva case atipice (capitolul 2 din `curs-04a`) cad în test, scorul se prăbușește. Agentul îți dă un singur număr, cu aerul că e „scorul modelului”. Întreabă-l cât de stabil este, sau cere validare încrucișată.

**Pentru client.** „Precizia estimată are ea însăși o marjă: o raportăm ca interval, nu ca o cifră exactă.”

## Capitolul 6. Ridge: coeficienți ținuți în frâu

**Concept.** Regresia obișnuită alege coeficienții cu cea mai mică eroare pe antrenare, oricât de mari ar fi (așa memorează). **Ridge** adaugă la eroare o penalizare pentru coeficienți mari:

$$\min_b \; \sum_i (y_i - \hat y_i)^2 + \alpha \sum_j b_j^2$$

Cu cât $\alpha$ e mai mare, cu atât coeficienții sunt strânși spre 0 și modelul memorează mai puțin; $\alpha = 0$ înseamnă regresia obișnuită. $\alpha$ nu se învață din date ca un coeficient: e un **hiperparametru**, ales prin validare încrucișată (`RidgeCV` încearcă o listă de valori și o păstrează pe cea mai bună). Penalizarea compară coeficienți între ei, deci toate coloanele (și cele 0/1 din one-hot) trebuie **standardizate** (modulul 3): `pregatire(scalare=True)`.

**Prezice.** Pe aceleași 150 de case pe care regresia obișnuită le-a memorat ($R^2$ test ≈ 0,38), ce va face Ridge?

**Cere agentului:**

> Pe primele 150 de case din antrenare, compară regresia liniară (în `Pipeline` cu completarea lipsurilor și one-hot) cu `RidgeCV` (alpha de la 0,1 la 100.000, pe scară logaritmică; în același `Pipeline`, plus standardizare). Raportează R² pe antrenare și pe test și alpha ales. Apoi desenează cum se schimbă coeficienții celor 6 predictori din capitolul 3 când alpha crește.

In [ ]:
# Cod de referință: Ridge pe cele 150 de case
from sklearn.linear_model import RidgeCV, Ridge

alfe = np.logspace(-1, 5, 61)          # 0,1 ... 100.000
X150, y150 = X_train[:150], y_train[:150]
evalueaza(make_pipeline(pregatire(), LinearRegression()), "liniară, 150 de case", X150, y150)
ridge150 = evalueaza(make_pipeline(pregatire(scalare=True), RidgeCV(alphas=alfe)), "Ridge, 150 de case", X150, y150)
print("alpha ales:", round(ridge150[-1].alpha_, 1))
pd.DataFrame(clasament)

In [ ]:
# Cod de referință: traseele coeficienților (6 predictori standardizați)
sase = ["GarageCars", "GarageArea", "GrLivArea", "OverallQual", "TotalBsmtSF", "YearBuilt"]
Z = StandardScaler().fit_transform(X_train[sase])
trasee = np.array([Ridge(alpha=a).fit(Z, y_train).coef_ for a in alfe])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alfe, trasee)
ax.set_xscale("log")
ax.set_xlabel("alpha (scară logaritmică)")
ax.set_ylabel("coeficient ($ pe o abatere standard)")
ax.set_title("Cu cât alpha crește, coeficienții se strâng spre 0")
ax.legend(sase, fontsize=8)
plt.tight_layout()
plt.show()

**Verifică.** Pe 150 de case: regresia obișnuită are $R^2$ test ≈ 0,38 (MAE ≈ 52.900 $), Ridge ≈ 0,85 (MAE ≈ 21.300 $; alpha ales ≈ 63). În grafic, toți coeficienții tind spre 0 când alpha crește; cei mari (`OverallQual`, `GrLivArea`) scad cel mai mult. `GarageArea` întâi **crește**: Ridge împarte efectul între cele două coloane corelate ale garajului (`GarageCars`, `GarageArea`) în loc să-l lase pe una singură.

**Capcană.** Fără standardizare, Ridge penalizează pe nedrept: un coeficient pe sqft e mic ca număr, unul pe „calitate de la 1 la 10” e mare, deci penalizarea îl lovește pe al doilea mai tare. Scalarea se face **în `Pipeline`**: dacă agentul standardizează tot tabelul înainte de împărțire, media și abaterea standard „văd” și casele de test (scurgere de informație).

**Pentru client.** „Pe puține date, un model ținut în frâu greșește mult mai puțin pe case noi decât unul lăsat liber.”

## Capitolul 7. Lasso: modelul își alege singur coloanele

**Concept.** **Lasso** penalizează suma valorilor absolute ale coeficienților:

$$\min_b \; \frac{1}{2n}\sum_i (y_i - \hat y_i)^2 + \alpha \sum_j |b_j|$$

Diferența față de Ridge: Lasso poate face coeficienți **exact 0**, adică scoate coloane din model. E util când avem sute de coloane și vrem un model pe care îl putem explica.

**Prezice.** Din cele 301 coloane obținute după one-hot, câte crezi că păstrează Lasso?

**Cere agentului:**

> Pe toată antrenarea, antrenează `LassoCV` (5 părți, `random_state=42`) în același `Pipeline` cu standardizare. Spune câți coeficienți sunt zero din total, R² pe test și cele 10 coloane cu cel mai mare coeficient în valoare absolută.

In [ ]:
# Cod de referință: Lasso pe toate casele
from sklearn.linear_model import LassoCV

lasso = evalueaza(make_pipeline(pregatire(scalare=True), LassoCV(cv=5, random_state=42, max_iter=50000)),
                  "Lasso, toate casele")
coef = pd.Series(lasso[-1].coef_, index=lasso[0].get_feature_names_out())
print(f"Coeficienți zero: {(coef == 0).sum()} din {len(coef)}    alpha ales: {lasso[-1].alpha_:,.0f}")
print(pd.DataFrame(clasament).tail(1).to_string(index=False))
print("\nCele mai importante 10 coloane ($ pe o abatere standard):")
coef[coef != 0].sort_values(key=abs, ascending=False).head(10).round(0)

**Verifică.** Lasso anulează 247 din 301 coeficienți și păstrează 54; $R^2$ pe test ≈ 0,888, MAE ≈ 17.114 $ (alpha ales ≈ 1.728). Pe primele locuri: `GrLivArea`, `RoofMatl_ClyTile`, `OverallQual`.

**Capcană.** `RoofMatl_ClyTile` (acoperiș din țiglă de argilă) pe locul doi? În antrenare există **o singură casă** cu acest acoperiș: casa 1299, cea uriașă vândută ieftin din capitolul 2. Lasso a folosit coloana ca pe o etichetă „aceasta e casa 1299”, ca să-i explice prețul atipic. O categorie rară cu coeficient mare nu e o regulă despre acoperișuri; verifică întotdeauna **câte case** stau în spatele unei coloane importante. Apoi: dintre doi predictori corelați, Lasso păstrează de obicei unul și îl anulează pe celălalt, uneori aproape la întâmplare. Un coeficient zero nu înseamnă „nu contează”, ci „altă coloană spune deja asta” (vezi `GarageCars` și `GarageArea`). Coeficienții sunt calculați pe date standardizate: îi comparăm între ei (importanță), dar nu se citesc în „dolari pe sqft”.

**Pentru client.** „Din cele 301 caracteristici, 54 sunt suficiente pentru aceeași precizie; cele mai importante sunt suprafața locuibilă și calitatea generală a casei.”

## Capitolul 8. Elastic Net și clasamentul

**Concept.** **Elastic Net** combină cele două penalizări:

$$\alpha \left[\, \rho \sum_j |b_j| + \frac{1-\rho}{2} \sum_j b_j^2 \,\right]$$

$\rho$ (`l1_ratio` în scikit-learn) merge de la 0 (Ridge) la 1 (Lasso). Din grupurile de predictori corelați păstrează mai mulți decât Lasso, dar poate și el elimina coloane. Același parametru `l1_ratio` alege tipul regularizării la **regresia logistică** din modulul 5.

**Prezice.** Care dintre cele trei va avea cea mai mică eroare pe test? Va fi diferența mare?

**Cere agentului:**

> Adaugă în clasament `RidgeCV` și `ElasticNetCV` (`l1_ratio` 0,2, 0,5 și 0,8; 5 părți; `random_state=42`) pe toată antrenarea, cu același `Pipeline` cu standardizare. Afișează clasamentul ordonat după MAE pe test.

In [ ]:
# Cod de referință: Ridge și Elastic Net pe toate casele, apoi clasamentul
from sklearn.linear_model import ElasticNetCV

evalueaza(make_pipeline(pregatire(scalare=True), RidgeCV(alphas=alfe)), "Ridge, toate casele")
# 1) cu lista implicită de valori alpha
enet = evalueaza(make_pipeline(pregatire(scalare=True),
                               ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8], cv=5, random_state=42, max_iter=50000)),
                 "Elastic Net, alpha implicit")
print(f"alpha ales: {enet[-1].alpha_:.1f}   valori încercate: de la {enet[-1].alphas_.min():.1f} la {enet[-1].alphas_.max():,.0f}")

# 2) cu o listă de valori alpha care coboară mult mai jos
enet = evalueaza(make_pipeline(pregatire(scalare=True),
                               ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8], alphas=np.logspace(-2, 4, 61),
                                            cv=5, random_state=42, max_iter=50000)),
                 "Elastic Net, alpha 0,01-10.000")
print(f"alpha ales: {enet[-1].alpha_:.2f}   l1_ratio ales: {enet[-1].l1_ratio_}")
pd.DataFrame(clasament).sort_values("MAE test")

**Verifică.** Cu lista implicită, Elastic Net alege alpha ≈ 75,8, **chiar valoarea cea mai mică încercată**, și are $R^2$ test 0,57. Cu lista 0,01-10.000: alpha ≈ 3,16, `l1_ratio` 0,8, MAE ≈ 18.667 $, $R^2$ 0,870. Ridge: MAE ≈ 18.742 $; Lasso: ≈ 17.114 $. Toate bat clar media (62.576 $).

**Capcană.** Dacă valoarea aleasă e **la capătul listei** încercate, înseamnă că lista era prea scurtă: cea mai bună valoare e probabil dincolo de ea. Lista implicită a lui `ElasticNetCV` e calculată din date și, aici, nu coboară destul. Agentul nu-ți spune asta: cere-i mereu „ce alpha a ales și între ce limite a căutat?”. Apoi: diferențele dintre cele trei sunt mici, de același ordin cu variația de la o împărțire la alta (capitolul 5). Iar dacă agentul **alege** modelul (sau `alpha`) după scorul pe test, testul nu mai e „nevăzut”: scorul raportat devine prea optimist. Alegem prin validare încrucișată pe antrenare; testul doar confirmă la final.

**Pentru client.** „Am încercat trei variante ale aceluiași model; rezultatele sunt apropiate, așa că o alegem pe cea mai ușor de explicat.”

## Ce îi spunem clientului

**Cere agentului:** „Scrie 3 propoziții pentru agenția imobiliară, cu cifrele din notebook: cât greșim în medie, ce interval de eroare putem promite pentru 8 case din 10 și ce caracteristici contează cel mai mult.”

Rezumatul de mai jos este calculat din variabilele notebook-ului; compară-l cu ce a scris agentul: **fiecare cifră din rezumat trebuie să apară și în rezultatele tale**.

In [ ]:
# Rezumat construit din rezultatele de mai sus (modelul Lasso: precis și ușor de explicat)
erori = (y_test - lasso.predict(X_test)).abs()
mae_media = clasament[0]["MAE test"]
# importanța, doar pentru caracteristicile prezente la cel puțin 1% din case (fără categorii rare)
prezenta = pd.Series((lasso[0][0].transform(X_train) != 0).mean(axis=0), index=coef.index)
importante = coef[(coef != 0) & (prezenta >= 0.01)].sort_values(key=abs, ascending=False).head(3).index
ro = lambda x: f"{x:,.0f}".replace(",", ".")   # 62576 -> „62.576”
print(f"1. Fără model, o estimare după medie greșește cu ~{ro(mae_media)} $ pe casă; modelul nostru greșește în medie cu ~{ro(erori.mean())} $.")
print(f"2. Pentru 8 case din 10 eroarea a fost sub {ro(erori.quantile(0.8))} $; pentru restul poate fi mult mai mare (case atipice).")
print(f"3. Modelul folosește {(coef != 0).sum()} din {len(coef)} caracteristici; cele mai importante: {', '.join(importante)}.")

---
# Referință

Definiții și detalii pentru acasă. Nu se parcurg la curs.

## R1. Cele trei regularizări în scikit-learn

| Model | Penalizarea | Ce face cu coeficienții | Clasa (cu alegere prin validare încrucișată) |
|---|---|---|---|
| Ridge | $\alpha \sum_j b_j^2$ (L2) | îi strânge spre 0, niciunul exact 0 | `Ridge` (`RidgeCV`) |
| Lasso | $\alpha \sum_j \lvert b_j\rvert$ (L1) | anulează o parte dintre ei (selecție de coloane) | `Lasso` (`LassoCV`) |
| Elastic Net | $\alpha[\rho\, L1 + \frac{1-\rho}{2} L2]$ | ambele; $\rho$ = `l1_ratio` | `ElasticNet` (`ElasticNetCV`) |

La **regresia logistică** (modulul 5) intensitatea se dă prin `C` $= 1/\alpha$ (C mic = regularizare puternică), iar tipul prin `l1_ratio` (0 = L2, 1 = L1).

**Parametru vs. hiperparametru.** Coeficienții $b_j$ sunt *parametri*: îi învață `fit` din date. $\alpha$ și `l1_ratio` sunt *hiperparametri*: îi alegem noi, înainte de `fit`, de obicei prin validare încrucișată.

## R2. Validarea încrucișată în 5 pași

1. Pune deoparte testul (20%) și nu-l mai atinge.
2. Împarte antrenarea în 5 părți egale.
3. Pentru fiecare valoare candidată a lui $\alpha$: antrenează de 5 ori (pe 4 părți) și măsoară eroarea pe partea rămasă; fă media.
4. Alege $\alpha$ cu cea mai mică eroare medie și reantrenează pe toată antrenarea (`RidgeCV`, `LassoCV`, `ElasticNetCV` fac pașii 2-4 singure).
5. Raportează eroarea **o singură dată**, pe test.

Pe larg, cu metrici și căutarea automată a hiperparametrilor: modulul 6.

## R3. De ce standardizăm înainte de regularizare

Penalizarea adună coeficienți ai unor coloane măsurate în unități diferite (sqft, ani, note de la 1 la 10). Coeficientul unei coloane cu valori mari e mic numeric și scapă de penalizare; al unei coloane cu valori mici e mare și e penalizat puternic. După standardizare toate coloanele au media 0 și abaterea standard 1, iar penalizarea le tratează la fel. Comparația dintre `StandardScaler` și `MinMaxScaler` pe date reale: `extra-04-lasso-ridge-detalii`.

## R4. Pentru aprofundare

- [extra-04-lasso-ridge-detalii.ipynb](extra-04-lasso-ridge-detalii.ipynb): Lasso și Ridge pe casele din King County, efectul standardizării vs. normalizării min-max asupra coeficienților, Elastic Net, grafice interactive.
- [extra-04-regresie-clasica.ipynb](extra-04-regresie-clasica.ipynb): regresia cu `statsmodels` (p-values, selecție pas cu pas).
- [extra-04-regresori-ml.ipynb](extra-04-regresori-ml.ipynb): aceleași probleme de regresie cu kNN, arbori, random forest și boosting (de citit după modulul 5).
- Slide-uri: [curs-04-regularizare.pdf](curs-04-regularizare.pdf), [curs-04-regularizare-lasso-ridge.pdf](curs-04-regularizare-lasso-ridge.pdf).

---
## Puntea curs → seminar

**Concepte cheie** (modulul 4)

1. **Ținta și împărțirea:** train/test o singură dată, la început; evaluăm doar pe test (nu pe un tabel fără țintă și nu pe antrenare).
2. **Modelul de referință** (media) și măsurile **MAE, RMSE** (în unitățile țintei) și **R²**.
3. **Coeficienții:** efectul unui predictor când ceilalți rămân la fel; depind de ce alte coloane sunt în model.
4. **Supraînvățarea:** bun pe antrenare, slab pe test; apare ușor cu multe coloane și puține rânduri. O singură împărțire poate fi norocoasă sau ghinionistă.
5. **Pipeline:** completarea lipsurilor, one-hot și scalarea se învață doar pe antrenare.
6. **Regularizarea:** Ridge strânge coeficienții, Lasso îi poate anula, Elastic Net le combină; `alpha` se alege prin validare încrucișată.

**Ce ceri asistentului AI** (exemplu de cerere)

> Pe tabelul `train` din `lab-set6`, fără `Id`, împarte 80/20 (`random_state=42`). Într-un `Pipeline` cu completarea lipsurilor, one-hot și standardizare, compară media (model de referință), regresia liniară, `RidgeCV`, `LassoCV` și `ElasticNetCV`. Raportează într-un singur tabel MAE și R² pe antrenare și pe test și spune câți coeficienți a anulat Lasso.

**Ce verifici înainte să accepți rezultatul**

- [ ] `Id` și ținta nu sunt printre predictori; evaluarea e pe partea de test din `train`, nu pe tabelul `test` fără preț;
- [ ] tabelul are un rând „model de referință”;
- [ ] pregătirea (lipsuri, one-hot, scalare) e în `Pipeline`, învățată doar pe antrenare;
- [ ] compari antrenarea cu testul (supraînvățare?);
- [ ] `alpha` a fost ales prin validare încrucișată, nu după scorul pe test;
- [ ] coeficienții au semn și mărime plauzibile pentru problemă.

Continuăm cu [seminar-04-regresie.ipynb](seminar-04-regresie.ipynb).

## Jurnal de sesiune

Completează la sfârșit (sau cere agentului: *„închide capitolul”*) și păstrează-l în copia ta a notebook-ului. Dacă agentul rămâne fără tokeni, lipește primerul și acest jurnal unui alt agent și scrie „continuă de aici”.

- Modul / capitolul la care am ajuns:
- Agent folosit și data:
- Ce am decis și de ce:
- Valori de control verificate (da / nu / care):
- Greșeli găsite la agent:
- Întrebare deschisă:
- Pentru următorul agent (ce a rămas de făcut, într-o propoziție):